# Q1

In [1]:
import os
import sys
import wget
from pyspark import SparkConf, SparkContext
# 设置 Python 解释器路径，防止 worker 连接超时
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# ------------------------------------------------------------
# 1. 初始化 SparkContext
# ------------------------------------------------------------
conf = SparkConf().setAppName("Sherlock_WordCount_Full").setMaster("local[*]")
sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

In [5]:
# 1. 파일 다운로드（文件下载）
file_name = "sherlock.txt"
target_url = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/sherlock.txt"

# 로컬에 파일이 이미 있는지 확인하다（检查本地是否已有文件，避免重复下载）
if not os.path.exists(file_name):
    print("Downloading text file...")
    wget.download(target_url, file_name)
    print("\nDownload finished！")

# 2. Spark 데이터 처리（Spark 数据处理）
# 파일 읽기（读取文件）
lines = sc.textFile(file_name)

# 계산 로직（计算逻辑）
q1_result = (
    lines.flatMap(lambda line: line.split(" "))       # 공백을 기준으로 단어를 분리하고 1차원 단어 집합으로 펼침 (按空格分词，打散成一维单词集合)
         .filter(lambda w: len(w) >= 2)               # 길이가 2 이상인 단어 필터링 (过滤出长度 >= 2 的单词)
         .map(lambda w: (w[:2].lower(), 1))           # 앞 2글자를 추출하여 소문자로 변환하고 (word, 1) 형태의 키-값 쌍으로 매핑 (截取前 2 个字母，转换为小写，并映射为 (word, 1) 的键值对)
         .reduceByKey(lambda a, b: a + b)             # 2-head 글자를 기준으로 그룹화하여 빈도수 계산 (按 2-head 字母分组并统计频次)
         .filter(lambda x: x[1] < 32)                 # 빈도수가 32 미만인 요소 필터링 (过滤出频次小于 32 的元素)
         .map(lambda x: ("total_sum", x[1]))          # 모든 요소의 Key를 강제로 "total_sum"으로 변경 (把所有元素的 Key 都变成 "total_sum")
         .reduceByKey(lambda a, b: a + b)             # 분산 환경에서 모든 빈도수를 더하여 최종 합계 계산 (将所有频次在分布式环境中相加，完成最终求和)
         .collect()                                   # 마지막으로 Action 연산자를 호출하여 계산 결과 가져오기 (最后调用 Action 算子拉取计算结果)
)

# 3. 결과 출력 (打印最终结果)
if q1_result:
    print("\nThe sum of frequencies for 2-head words with a frequency less than 32 is:", q1_result[0][1])
else:
    print("\nNo words found matching the criteria.")

100% [............................................................................] 243953 / 243953
Download finished！

The sum of frequencies for 2-head words with a frequency less than 32 is: 830


# Q2

In [6]:
# 1. 파일 다운로드 (文件下载)
file_name_q2 = "int-pair-input.txt"
target_url_q2 = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/int-pair-input.txt"

# 로컬에 파일이 이미 있는지 확인하다 (检查本地是否已有文件，避免重复下载)
if not os.path.exists(file_name_q2):
    print("Downloading text file...")
    wget.download(target_url_q2, file_name_q2)
    print("\nDownload finished!")

# 각 줄에서 첫 번째 숫자만 추출 (提取每一行的第一个数字)
def parse_first_element(line):
    # 뒤에서 나누기 쉽도록 괄호를 쉼표로 바꾸다 (把括号换成逗号，以便后面分割)
    clean_line = line.replace("(", "").replace(")", "")
    # 쉼표로 문자열을 구분하여 리스트로 변환하다（以逗号分割字符串，并改为列表）
    parts = clean_line.split(",")
    # 리스트가 비어있지 않은지 확인하다 (检查列表是否不为空)
    if parts != []:
        return int(parts[0])
    return None

# 2. Spark 데이터 처리 (Spark 数据处理)
# 파일을 읽고 첫 번째 숫자만 추출하여 RDD 생성 (读取文件并映射出第一个数字生成 RDD)
rdd_q2 = sc.textFile(file_name_q2).map(parse_first_element).filter(lambda x: x is not None)

# 지표 1: 첫 번째 정수 쌍의 첫 번째 원소 (指标 1: 第一个整数对的第一个元素)
val1 = rdd_q2.collect()[0]

# 지표 2: 모든 정수 쌍의 첫 번째 원소 중 최댓값 (指标 2: 所有第一个元素中的最大值)
# sortByKey(False)를 사용하여 내림차순 정렬 후 첫 번째 값 추출 (使用 sortByKey 降序排列并提取第一个值)
val2 = rdd_q2.map(lambda x: (x, 1)).sortByKey(False).collect()[0][0]

# 지표 3: 모든 정수 쌍의 첫 번째 원소 중 최솟값 (指标 3: 所有第一个元素中的最小值)
# sortByKey(True)를 사용하여 오름차순 정렬 후 첫 번째 값 추출 (使用 sortByKey 升序排列并提取第一个值)
val3 = rdd_q2.map(lambda x: (x, 1)).sortByKey(True).collect()[0][0]

# 지표 4: 첫 번째 원소가 1인 정수 쌍의 개수 (指标 4: 第一个元素为 1 的个数)
# filter로 1만 남기고 reduceByKey로 개수 계산 (用 filter 找出1，再用 reduceByKey 计算个数)
val4_list = rdd_q2.filter(lambda x: x == 1).map(lambda x: ("count", 1)).reduceByKey(lambda a, b: a + b).collect()
val4 = val4_list[0][1] if val4_list else 0

# 지표 5: 첫 번째 원소가 2인 정수 쌍의 개수 (指标 5: 第一个元素为 2 的个数)
val5_list = rdd_q2.filter(lambda x: x == 2).map(lambda x: ("count", 1)).reduceByKey(lambda a, b: a + b).collect()
val5 = val5_list[0][1] if val5_list else 0

# 3. 결과 출력 (打印最终结果)
total_sum = val1 + val2 + val3 + val4 + val5
print(f"\nThe calculated values are:")
print(f"- 1st element of the 1st pair: {val1}")
print(f"- Maximum value: {val2}")
print(f"- Minimum value: {val3}")
print(f"- Count of pairs starting with 1: {val4}")
print(f"- Count of pairs starting with 2: {val5}")
print(f"\nThe total sum of these five numbers is: {total_sum}")

100% [................................................................................] 8730 / 8730
Download finished!

The calculated values are:
- 1st element of the 1st pair: 2
- Maximum value: 32
- Minimum value: 0
- Count of pairs starting with 1: 31
- Count of pairs starting with 2: 28

The total sum of these five numbers is: 93


# Q3

In [4]:
# 1. 5년치 데이터 파일 다운로드 (下载 5 年数据)
years = ["2015", "2016", "2017", "2018", "2019"]
for year in years:
    file_name = f"{year}.csv"
    target_url = f"https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World Hapiness Report/{year}.csv"
    # 이미 로컬에 있는 파일은 건너뜀 (本地已存在则跳过，避免重复下载)
    if not os.path.exists(file_name):
        wget.download(target_url, out=file_name)
print("\nAll files downloaded!")


# 2. 데이터 전처리(数据清洗)
def parse_csv(line):
    # 헤더를 읽으면 None을 반환해서 버림 (读到表头就返回 None 扔掉，因为表头里有 "Country" 这个单词)
    if "Country" in line:
        return None

    # 쉼표기준으로 분리 (按照逗号切开)
    parts = line.split(",")

    # 0번째와 2번째 값만 가져옴 (拿第 0 个和第 2 个)
    country = parts[0]
    score = float(parts[2])

    return (country, score)


# 3. 데이터를 하나씩 읽음 (读取每一年的数据)
rdd15 = sc.textFile("2015.csv").map(parse_csv).filter(lambda x: x is not None)
rdd16 = sc.textFile("2016.csv").map(parse_csv).filter(lambda x: x is not None)
rdd17 = sc.textFile("2017.csv").map(parse_csv).filter(lambda x: x is not None)
rdd18 = sc.textFile("2018.csv").map(parse_csv).filter(lambda x: x is not None)
rdd19 = sc.textFile("2019.csv").map(parse_csv).filter(lambda x: x is not None)

# 4. 연속으로 Join해서 하나로 합침 (连续 Join 拼在一起)
joined_rdd = rdd15.join(rdd16).join(rdd17).join(rdd18).join(rdd19)


# 5. 점수를 하나씩 꺼냄 (一步步地把连环括号里的分数取出来)
def check_increase(record):
    # 이때 record[1]의 모양은 ((((점수15, 점수16), 점수17), 점수18), 점수19) (此时 record[1] 的长相是 ((((分15, 分16), 分17), 分18), 分19))
    scores_tuple = record[1]

    part1 = scores_tuple[0]  # (((점수15, 점수16), 점수17), 점수18) 를 얻음 (得到 (((分15, 分16), 分17), 分18))
    s19 = scores_tuple[1]    # 2019년 점수를 얻음 (拿到 2019 年的分数)

    part2 = part1[0]         # ((점수15, 점수16), 점수17) 를 얻음 (得到 ((分15, 分16), 分17))
    s18 = part1[1]           # 2018년 점수를 얻음 (拿到 2018 年的分数)

    part3 = part2[0]         # (점수15, 점수16) 을 얻음 (得到 (分15, 分16))
    s17 = part2[1]           # 2017년 점수를 얻음 (拿到 2017 年的分数)

    s15 = part3[0]           # 2015년 점수를 얻음 (拿到 2015 年的分数)
    s16 = part3[1]           # 2016년 점수를 얻음 (拿到 2016 年的分数)

    # if 판단(if 判断)
    if s15 <= s16 and s16 <= s17 and s17 <= s18 and s18 <= s19:
        return True
    else:
        return False


# 6. 조건에 맞는 국가를 필터링하고 이름을 뽑아 collect (过滤符合条件的国家，提取名字并 collect)
result_q3 = joined_rdd.filter(check_increase).map(lambda x: x[0]).collect()

# 7. 결과 출력 (打印结果)
print("\nCountries with a continuous increase or identical happiness score over 5 years:")
for country in result_q3:
    print(f"- {country}")

100% [................................................................................] 7732 / 7732
All files downloaded!

Countries with a continuous increase or identical happiness score over 5 years:
- Finland
- Poland
- Cambodia
- Congo (Brazzaville)
- Niger
- Benin
- Senegal
- Burkina Faso
- Portugal
- Gabon
- Estonia
- Ivory Coast
- Malta
- Czech Republic
- Philippines
- Honduras
- Slovakia
- Romania
- Serbia
- Mongolia
- Chad
- Syria
- Cameroon
- Togo
- Hungary
- Latvia
- Dominican Republic
- Tajikistan
